# IFCNetCore — extract DINOv3 + SigLIP2 + DuoDuoCLIP visual embeddings

**Inputs** — only one file in your Drive: `IFCNetCorePng.zip` (the 445 MB renders zip).
Everything else is generated / cloned / downloaded inside this notebook.

**Outputs** — one zip per encoder back to Drive:
- `dinov3_ifcnet_colorless.zip`
- `siglip_ifcnet_colorless.zip`
- `duoduo_ifcnet_colorless.zip`

Each contains 7930 `.npy` files (one 1024-d embedding per object), under
`features/{obj_id}.npy`.

Pipeline per encoder: load 12 stock renders per object → encoder → mean-pool
across views (DINOv3 / SigLIP) or internal multi-view pooling (DuoDuoCLIP) →
save one .npy per object.

Each cell is **resume-safe** — re-running skips obj_ids whose `.npy` already
exists, so a disconnect doesn't waste work.

**Pick a GPU runtime** before running anything: Runtime → Change runtime type → T4/L4/A100.

## 0. Setup — Drive, GPU sanity, paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
import os
from pathlib import Path

# Adjust this if your zip lives elsewhere in Drive:
DRIVE_ZIP   = Path('/content/drive/MyDrive/IFCNetCorePng.zip')
DRIVE_OUT   = Path('/content/drive/MyDrive')    # where to dump result zips

RENDERS_ROOT = Path('/content/data/IFCNetCore/renders')
FEATURES_ROOT = Path('/content/data/IFCNetCore/processed/rendered_features')
METADATA_PATH = Path('/content/data/IFCNetCore/metadata.json')

assert DRIVE_ZIP.exists(), f'Expected {DRIVE_ZIP} on Drive — upload IFCNetCorePng.zip there first.'
print('zip   :', DRIVE_ZIP, '(', DRIVE_ZIP.stat().st_size // (1024*1024), 'MB)')
print('drive :', DRIVE_OUT)

## 1. Unzip renders + generate metadata.json on the fly

In [ ]:
RENDERS_ROOT.mkdir(parents=True, exist_ok=True)
!unzip -q -n "$DRIVE_ZIP" -d "$RENDERS_ROOT"
n_png = sum(1 for _ in RENDERS_ROOT.rglob('*.png'))
print(f'unzipped: {n_png} PNGs')

In [ ]:
# Walk the unzipped tree and write metadata.json — one row per object.
import json

rows = []
for class_dir in sorted(RENDERS_ROOT.iterdir()):
    if not class_dir.is_dir():
        continue
    for split_dir in sorted(class_dir.iterdir()):
        if not split_dir.is_dir():
            continue
        # PNGs are named {hash}.{view_idx}.png. Collapse to unique obj_ids.
        obj_ids = sorted({p.stem.rsplit('.', 1)[0] for p in split_dir.glob('*.png')})
        for oid in obj_ids:
            views = sorted(split_dir.glob(f'{oid}.*.png'))
            rows.append({
                'obj_id': oid,
                'ifc_class': class_dir.name,
                'split': split_dir.name,
                'render_dir': str(split_dir.relative_to(RENDERS_ROOT.parent.parent)),
                'num_renders': len(views),
            })

METADATA_PATH.parent.mkdir(parents=True, exist_ok=True)
METADATA_PATH.write_text(json.dumps(rows, indent=2))
print(f'{len(rows)} objects across {len({r["ifc_class"] for r in rows})} classes')
print(f'render counts — min={min(r["num_renders"] for r in rows)} '
      f'max={max(r["num_renders"] for r in rows)}')
print(f'wrote → {METADATA_PATH}')

## 2. Shared helpers (same as `_ifcnet_views.py`)

In [ ]:
import json
import shutil
import subprocess
from dataclasses import dataclass
from pathlib import Path
from tqdm import tqdm
import numpy as np


@dataclass(frozen=True)
class ObjectEntry:
    obj_id: str
    ifc_class: str
    split: str
    views: list


def load_entries(metadata_path: Path, renders_root: Path) -> list:
    rows = json.loads(metadata_path.read_text())
    out = []
    for r in rows:
        render_dir = renders_root / r['ifc_class'] / r['split']
        views = sorted(render_dir.glob(f"{r['obj_id']}.*.png"))
        out.append(ObjectEntry(r['obj_id'], r['ifc_class'], r['split'], views))
    return out


def filter_pending(entries: list, out_dir: Path) -> list:
    return [e for e in entries if not (out_dir / f'{e.obj_id}.npy').exists()]


def zip_and_upload(features_subdir: Path, archive_name: str, drive_out: Path) -> Path:
    """Zip the colorless/ folder under features_subdir and copy the archive to Drive.

    Robust replacement for the older `!cd "{...}" && zip ... && cp ...` shell magic:
      - errors surface as Python exceptions (subprocess.check / shutil.copy2)
      - always overwrites the local zip so a re-run produces a complete archive
        reflecting whatever .npy files are now on disk
      - safe to call mid-extraction as a checkpoint (e.g. after a partial run)
    """
    src_dir = features_subdir / 'colorless'
    assert features_subdir.is_dir(), f'missing {features_subdir}'
    n_files = sum(1 for _ in src_dir.glob('*.npy'))
    if n_files == 0:
        print(f'  [warn] {src_dir} has 0 .npy files — skipping zip')
        return None

    local_zip = Path('/content') / archive_name
    if local_zip.exists():
        local_zip.unlink()
    subprocess.run(['zip', '-qr', str(local_zip), 'colorless'],
                   cwd=str(features_subdir), check=True)
    drive_out.mkdir(parents=True, exist_ok=True)
    drive_zip = drive_out / archive_name
    shutil.copy2(local_zip, drive_zip)
    size_mb = drive_zip.stat().st_size / (1024 * 1024)
    print(f'  ✓ zipped {n_files} files → {drive_zip}  ({size_mb:.1f} MB)')
    return drive_zip


ENTRIES = load_entries(METADATA_PATH, RENDERS_ROOT)
print(f'loaded {len(ENTRIES)} object entries')

## 3. SigLIP2 large-patch16-512 (~1.2 h on T4 for 7930 objects)

Easiest one — pure HuggingFace, no repo cloning.

In [ ]:
!pip install -q "transformers>=4.49" pillow

In [ ]:
import torch
from PIL import Image
from transformers import AutoModel, AutoProcessor

SIGLIP_MODEL_ID = 'google/siglip2-large-patch16-512'
SIGLIP_OUT = FEATURES_ROOT / 'siglip2_large_16_512' / 'colorless'
SIGLIP_OUT.mkdir(parents=True, exist_ok=True)


def _to_tensor(out):
    """Unwrap whatever .get_image_features() returns into a (N, D) tensor.

    Different transformers versions return different types here:
      - a bare Tensor (older / direct image-tower call)
      - a BaseModelOutputWithPooling (newer, has .pooler_output)
      - a CLIPVisionModelOutput-style object (has .image_embeds)
    """
    if isinstance(out, torch.Tensor):
        return out
    for attr in ('image_embeds', 'pooler_output', 'last_hidden_state'):
        if hasattr(out, attr):
            t = getattr(out, attr)
            return t.mean(dim=1) if attr == 'last_hidden_state' and t.ndim == 3 else t
    raise TypeError(f'cannot extract tensor from {type(out).__name__}: {out}')


pending = filter_pending(ENTRIES, SIGLIP_OUT)
print(f'[siglip] {len(pending)} / {len(ENTRIES)} pending (rest already done)')

if pending:
    siglip = AutoModel.from_pretrained(SIGLIP_MODEL_ID).cuda().eval()
    siglip_proc = AutoProcessor.from_pretrained(SIGLIP_MODEL_ID)

    # One-time smoke test to surface tensor-shape issues before the long loop.
    with torch.no_grad():
        _smoke_pil = Image.open(pending[0].views[0]).convert('RGB')
        _smoke_in = siglip_proc(images=[_smoke_pil], return_tensors='pt')['pixel_values'].cuda()
        _smoke_raw = siglip.get_image_features(_smoke_in)
        _smoke_t = _to_tensor(_smoke_raw)
    print(f'[siglip] smoke: raw type={type(_smoke_raw).__name__}  tensor shape={tuple(_smoke_t.shape)}')

    for e in tqdm(pending, desc='siglip'):
        try:
            pil_list = [Image.open(v).convert('RGB') for v in e.views]
            inputs = siglip_proc(images=pil_list, padding=True, truncation=True, return_tensors='pt')
            with torch.no_grad():
                raw = siglip.get_image_features(inputs['pixel_values'].cuda())
                feats = _to_tensor(raw)               # (N_views, D)
            vec = feats.mean(dim=0).cpu().numpy().astype(np.float32)
            np.save(SIGLIP_OUT / f'{e.obj_id}.npy', vec)
        except Exception as exc:
            print(f'  [error] {e.obj_id}: {exc}')

    # Free GPU before the next encoder
    del siglip, siglip_proc
    torch.cuda.empty_cache()

print('done:', len(list(SIGLIP_OUT.glob('*.npy'))), 'embeddings written')

In [ ]:
zip_and_upload(FEATURES_ROOT / 'siglip2_large_16_512',
               archive_name='siglip_ifcnet_colorless.zip',
               drive_out=DRIVE_OUT)

## 4. DINOv3 ViT-L/16 (~3.5 h on T4 for 7930 objects)

In [ ]:
# Clone the official repo (provides hubconf for torch.hub.load)
if not Path('/content/dinov3').exists():
    !git clone -q https://github.com/facebookresearch/dinov3.git /content/dinov3
!ls /content/dinov3 | head

In [ ]:
# Download the ViT-L/16 weights. IMPORTANT — keep the filename exactly as below.
# The hubconf validates the hash pattern in the filename; renaming to model.pth fails.
DINOV3_WEIGHTS = Path('/content/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth')
if not DINOV3_WEIGHTS.exists():
    !wget -q -O "$DINOV3_WEIGHTS" https://huggingface.co/jaychempan/dinov3/resolve/44126792d766b593994f73c1019d7788a2a715e6/dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth
print(f'weights: {DINOV3_WEIGHTS.stat().st_size // (1024*1024)} MB')

In [ ]:
import torch
import torchvision.transforms.functional as TF
from PIL import Image

PATCH_SIZE = 16
IMAGE_SIZE = 768
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

DINOV3_OUT = FEATURES_ROOT / 'dinov3' / 'colorless'
DINOV3_OUT.mkdir(parents=True, exist_ok=True)


def resize_for_dinov3(pil):
    w, h = pil.size
    h_patches = int(IMAGE_SIZE / PATCH_SIZE)
    w_patches = int((w * IMAGE_SIZE) / (h * PATCH_SIZE))
    resized = TF.resize(pil, (h_patches * PATCH_SIZE, w_patches * PATCH_SIZE))
    tens = TF.to_tensor(resized)
    return TF.normalize(tens, mean=IMAGENET_MEAN, std=IMAGENET_STD)


pending = filter_pending(ENTRIES, DINOV3_OUT)
print(f'[dinov3] {len(pending)} / {len(ENTRIES)} pending')

if pending:
    dinov3 = torch.hub.load('/content/dinov3', 'dinov3_vitl16',
                            source='local',
                            weights=str(DINOV3_WEIGHTS),
                            skip_validation=True)
    dinov3 = dinov3.cuda().eval()

    for e in tqdm(pending, desc='dinov3'):
        try:
            tensors = [resize_for_dinov3(Image.open(v).convert('RGB')).unsqueeze(0)
                       for v in e.views]
            batch = torch.cat(tensors).cuda()
            with torch.no_grad():
                feats = dinov3(batch)
            vec = feats.mean(dim=0).cpu().numpy().astype(np.float32)
            np.save(DINOV3_OUT / f'{e.obj_id}.npy', vec)
        except Exception as exc:
            print(f'  [error] {e.obj_id}: {exc}')

    del dinov3
    torch.cuda.empty_cache()

print('done:', len(list(DINOV3_OUT.glob('*.npy'))), 'embeddings written')

In [ ]:
zip_and_upload(FEATURES_ROOT / 'dinov3',
               archive_name='dinov3_ifcnet_colorless.zip',
               drive_out=DRIVE_OUT)

## 5. DuoDuoCLIP (~1.5 h on T4 for 7930 objects)

Unlike the previous two, `encode_image` accepts all 12 views at once and pools
internally — no extra mean-pool afterwards.

In [ ]:
if not Path('/content/DuoduoCLIP').exists():
    !git clone -q https://github.com/3dlg-hcvc/DuoduoCLIP.git /content/DuoduoCLIP
!pip install -q /content/DuoduoCLIP/open_clip_mod/

In [ ]:
import sys, torch
from PIL import Image
sys.path.insert(0, '/content/DuoduoCLIP')
from src.model.wrapper import get_model

DUODUO_OUT = FEATURES_ROOT / 'duoduo' / 'colorless'
DUODUO_OUT.mkdir(parents=True, exist_ok=True)
DUODUO_CKPT = 'Four_1to6F_bs1600_LT6.ckpt'
DUODUO_RES = 224

pending = filter_pending(ENTRIES, DUODUO_OUT)
print(f'[duoduo] {len(pending)} / {len(ENTRIES)} pending')

if pending:
    duoduo = get_model(DUODUO_CKPT, device='cuda')

    for e in tqdm(pending, desc='duoduo'):
        try:
            imgs = []
            for v in e.views:
                pil = Image.open(v).convert('RGB').resize((DUODUO_RES, DUODUO_RES))
                imgs.append(np.expand_dims(np.asarray(pil), 0))
            batch = np.concatenate(imgs, axis=0)
            with torch.no_grad():
                feats = duoduo.encode_image(batch)
            vec = feats.cpu().numpy().reshape(-1).astype(np.float32)
            np.save(DUODUO_OUT / f'{e.obj_id}.npy', vec)
        except Exception as exc:
            print(f'  [error] {e.obj_id}: {exc}')

    del duoduo
    torch.cuda.empty_cache()

print('done:', len(list(DUODUO_OUT.glob('*.npy'))), 'embeddings written')

In [ ]:
zip_and_upload(FEATURES_ROOT / 'duoduo',
               archive_name='duoduo_ifcnet_colorless.zip',
               drive_out=DRIVE_OUT)

## 6. Final tally

In [ ]:
for enc in ['siglip2_large_16_512', 'dinov3', 'duoduo']:
    d = FEATURES_ROOT / enc / 'colorless'
    n = len(list(d.glob('*.npy'))) if d.exists() else 0
    sample = list(d.glob('*.npy'))[:1]
    dim = np.load(sample[0]).shape if sample else None
    print(f'  {enc:25s}  {n:>5d} files  dim={dim}')